# LAB – Clustering (IA não supervisionada): K-Means com o dataset Country Data

**PUC-SP – Ciência da Computação – Inteligência Artificial (IA ML)**

**Alunos (dupla):**
- João Avila Harduin - RA00332195
- Kauã Cordeiro Cavalheiro - RA00331660

**Objetivo:** experimentar e explorar um modelo de *clustering* (agrupamento) com Scikit-Learn: o **K-Means** (`KMeans`).

**Dataset:** [Unsupervised Learning on Country Data](https://www.kaggle.com/datasets/rohan0301/unsupervised-learning-on-country-data) do Kaggle.
São indicadores socioeconômicos e de saúde de 167 países (mortalidade infantil, renda, PIB per capita, expectativa de vida etc.).

**Problema:** aprendizado **não supervisionado**. Não existe uma coluna "resposta". O objetivo é descobrir grupos de países com perfis
parecidos, por exemplo para ajudar uma ONG a decidir quais países mais precisam de ajuda humanitária.

## 1. Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

RANDOM_STATE = 42
CORES = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#DD8452", "#937860", "#DA8BC3", "#8C8C8C", "#CCB974", "#64B5CD"]

## 2. Carregando os dados

Baixe o arquivo `Country-data.csv` na página do dataset no Kaggle e coloque na mesma pasta do notebook.
Se o arquivo não for encontrado, o código tenta baixar automaticamente com o pacote `kagglehub` (precisa de `pip install kagglehub`).

In [ ]:
CAMINHO = "Country-data.csv"

try:
    df = pd.read_csv(CAMINHO)
except FileNotFoundError:
    import kagglehub
    pasta = kagglehub.dataset_download("rohan0301/unsupervised-learning-on-country-data")
    df = pd.read_csv(f"{pasta}/Country-data.csv")

print("Dimensões:", df.shape)
df.head()

**Dicionário das colunas:**

| Coluna | Significado |
|---|---|
| `country` | Nome do país |
| `child_mort` | Mortes de crianças menores de 5 anos a cada 1000 nascidos vivos |
| `exports` | Exportações (% do PIB) |
| `health` | Gasto com saúde (% do PIB) |
| `imports` | Importações (% do PIB) |
| `income` | Renda líquida por pessoa |
| `inflation` | Inflação anual (%) |
| `life_expec` | Expectativa de vida (anos) |
| `total_fer` | Número médio de filhos por mulher |
| `gdpp` | PIB per capita |

## 3. Análise exploratória

In [ ]:
df.info()

In [ ]:
print("Valores ausentes por coluna:")
print(df.isna().sum())
print("\nPaíses duplicados:", df["country"].duplicated().sum())

In [ ]:
df.describe().round(2)

Observações:
- O dataset não tem valores ausentes, então não precisamos de `SimpleImputer`.
- As variáveis estão em **escalas muito diferentes**: `income` e `gdpp` chegam a dezenas de milhares, enquanto `total_fer` fica entre 1 e 8.

In [ ]:
features = ["child_mort", "exports", "health", "imports", "income",
            "inflation", "life_expec", "total_fer", "gdpp"]

fig, axes = plt.subplots(3, 3, figsize=(13, 9))
for ax, col in zip(axes.ravel(), features):
    ax.hist(df[col], bins=25, color="#4C72B0", edgecolor="white")
    ax.set_title(col)
plt.suptitle("Distribuição de cada variável")
plt.tight_layout()
plt.show()

print("Assimetria (skewness) de cada variável:")
print(df[features].skew().round(2).sort_values(ascending=False))

Várias variáveis (`income`, `gdpp`, `child_mort`, `exports`, `imports`, `inflation`) têm **cauda longa à direita**: poucos países
muito ricos ou com valores extremos. Isso atrapalha modelos baseados em distância, porque esses poucos países "puxam" os centros dos grupos.

In [ ]:
corr = df[features].corr()

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(features)), features, rotation=45, ha="right")
ax.set_yticks(range(len(features)), features)
for i in range(len(features)):
    for j in range(len(features)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im)
ax.set_title("Correlação entre as variáveis")
plt.tight_layout()
plt.show()

A matriz mostra grupos de variáveis que andam juntas:
- `income` e `gdpp` são fortemente correlacionadas (países ricos têm renda alta).
- `child_mort` e `total_fer` sobem juntas e caem quando `life_expec` sobe.

Ou seja, existe uma dimensão principal de **"nível de desenvolvimento"** nos dados, o que sugere que os países vão formar grupos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(df["gdpp"], df["child_mort"], alpha=0.7, c="#4C72B0")
axes[0].set_xlabel("PIB per capita"); axes[0].set_ylabel("Mortalidade infantil")
axes[1].scatter(df["income"], df["life_expec"], alpha=0.7, c="#55A868")
axes[1].set_xlabel("Renda por pessoa"); axes[1].set_ylabel("Expectativa de vida")
plt.suptitle("Sem rótulos: será que existem grupos aqui?")
plt.tight_layout()
plt.show()

## 4. Pré-processamento

- **`country`** é só um identificador, então fica de fora do modelo (mas guardamos para interpretar os grupos depois).
- **Transformação logarítmica (`log1p`)** nas variáveis com cauda longa e sem valores negativos, para reduzir o peso dos extremos.
  A `inflation` tem valores negativos, então não aplicamos log nela.
- **Padronização (`StandardScaler`)**: obrigatória aqui. No LAB anterior, as árvores de decisão não precisavam de escala, mas
  o K-Means usa **distância euclidiana**. Sem padronizar, `gdpp` (valores na casa dos milhares)
  dominaria totalmente o cálculo e as outras variáveis seriam ignoradas.

In [ ]:
features_log = ["child_mort", "exports", "imports", "income", "gdpp"]

X = df[features].copy()
X[features_log] = np.log1p(X[features_log])

escalador = StandardScaler()
X_scaled = escalador.fit_transform(X)

print("Assimetria depois do log:")
print(X[features_log].skew().round(2))
print("\nMédia depois da padronização:", X_scaled.mean(axis=0).round(2))
print("Desvio padrão depois da padronização:", X_scaled.std(axis=0).round(2))

### PCA para visualização

Temos 9 dimensões e não dá para desenhar isso. Usamos **PCA** só para projetar os dados em 2 dimensões e conseguir **ver** os grupos.
O clustering é feito com as 9 variáveis padronizadas, e não com a projeção.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

print("Variância explicada por componente:", pca.explained_variance_ratio_.round(3))
print(f"Total explicado pelas 2 componentes: {pca.explained_variance_ratio_.sum():.1%}")

plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.7, c="#8C8C8C")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("Países projetados em 2D com PCA (sem clusters)")
plt.show()

### Funções auxiliares

Para não repetir código, criamos funções para: desenhar os grupos no plano do PCA, calcular as métricas de qualidade e
dar nomes aos grupos de acordo com o PIB per capita médio.

In [ ]:
def plot_clusters(labels, titulo, nomes=None):
    fig, ax = plt.subplots(figsize=(8, 5.5))
    for i, grupo in enumerate(np.unique(labels)):
        mascara = labels == grupo
        rotulo = nomes[grupo] if nomes else f"Grupo {grupo}"
        ax.scatter(X_pca[mascara, 0], X_pca[mascara, 1], label=rotulo, alpha=0.75, c=CORES[i % len(CORES)])
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    ax.set_title(titulo)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


def metricas(labels):
    return {
        "Nº de grupos": len(np.unique(labels)),
        "Silhouette (↑)": silhouette_score(X_scaled, labels),
        "Davies-Bouldin (↓)": davies_bouldin_score(X_scaled, labels),
        "Calinski-Harabasz (↑)": calinski_harabasz_score(X_scaled, labels),
    }


def renumerar_por_pib(labels):
    # Renumera os grupos para que 0 = menor PIB per capita médio e o último = maior.
    ordem = df.groupby(labels)["gdpp"].mean().sort_values().index
    mapa = {antigo: novo for novo, antigo in enumerate(ordem)}
    return np.array([mapa[l] for l in labels])

**Métricas de qualidade de clusters** (não precisam de rótulo verdadeiro):
- **Silhouette** (de -1 a 1): mede se cada ponto está mais perto do próprio grupo do que do grupo vizinho. Quanto maior, melhor.
- **Davies-Bouldin**: compara o espalhamento dentro dos grupos com a distância entre eles. Quanto **menor**, melhor.
- **Calinski-Harabasz**: razão entre a dispersão entre grupos e dentro dos grupos. Quanto maior, melhor.

## 5. Modelo – K-Means (`KMeans`)

**Como funciona:**
1. Escolhemos o número de grupos `K` e o algoritmo sorteia `K` centros iniciais (com a estratégia `k-means++`, que espalha bem os centros).
2. Cada país é atribuído ao **centro mais próximo**.
3. Cada centro é recalculado como a **média** dos países do seu grupo.
4. Os passos 2 e 3 se repetem até os centros pararem de mudar.

O K-Means minimiza a **inércia**: a soma das distâncias ao quadrado de cada ponto até o centro do seu grupo.
O ponto fraco é que **precisamos escolher K** antes de treinar.

### 5.1 Escolhendo K: método do cotovelo e silhouette

In [ ]:
valores_k = range(2, 11)
inercias, silhuetas = [], []

for k in valores_k:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    rotulos = km.fit_predict(X_scaled)
    inercias.append(km.inertia_)
    silhuetas.append(silhouette_score(X_scaled, rotulos))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(valores_k), inercias, "o-", color="#4C72B0")
axes[0].set_xlabel("K"); axes[0].set_ylabel("Inércia")
axes[0].set_title("Método do cotovelo")
axes[1].plot(list(valores_k), silhuetas, "o-", color="#DD8452")
axes[1].set_xlabel("K"); axes[1].set_ylabel("Silhouette")
axes[1].set_title("Silhouette médio por K")
plt.tight_layout()
plt.show()

pd.DataFrame({"K": list(valores_k), "Inércia": inercias, "Silhouette": silhuetas}).set_index("K").round(3)

**Como ler os gráficos:**
- No **cotovelo**, a inércia sempre cai quando K aumenta (mais grupos = pontos mais perto de algum centro). Procuramos o ponto em que a
  queda deixa de ser grande. Neste dataset a curva costuma "dobrar" por volta de **K = 3**.
- O **silhouette** normalmente é maior em K = 2, porque separar só "ricos" e "pobres" é a divisão mais óbvia. Porém, 2 grupos é pouco
  informativo para o problema, e K = 3 mantém um silhouette próximo ao máximo.

Escolhemos **K = 3**, que equilibra as duas métricas e é fácil de interpretar: países de **baixo**, **médio** e **alto** desenvolvimento.

In [ ]:
K = 3

kmeans = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE)
rotulos_km = renumerar_por_pib(kmeans.fit_predict(X_scaled))

NOMES = {0: "Baixo desenvolvimento", 1: "Desenvolvimento intermediário", 2: "Alto desenvolvimento"} if K == 3 else None

df["cluster_kmeans"] = rotulos_km
print("Países por grupo:")
print(df["cluster_kmeans"].map(NOMES or (lambda g: f"Grupo {g}")).value_counts())
print()
for nome, valor in metricas(rotulos_km).items():
    print(f"{nome}: {valor:.3f}" if isinstance(valor, float) else f"{nome}: {valor}")

In [ ]:
plot_clusters(rotulos_km, "K-Means (K = 3) no plano do PCA", NOMES)

### 5.2 Interpretando os grupos

O clustering só é útil se conseguirmos explicar **o que caracteriza** cada grupo. Para isso olhamos a média de cada variável
nos **valores originais** (sem log e sem padronização).

In [ ]:
perfil = df.groupby("cluster_kmeans")[features].mean().round(1)
if NOMES:
    perfil.index = perfil.index.map(NOMES)
perfil

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, col in zip(axes, ["child_mort", "life_expec", "income", "total_fer"]):
    dados = [df.loc[df["cluster_kmeans"] == g, col] for g in range(K)]
    caixa = ax.boxplot(dados, patch_artist=True)
    for patch, cor in zip(caixa["boxes"], CORES):
        patch.set_facecolor(cor)
    ax.set_xticks(range(1, K + 1), ["Baixo", "Médio", "Alto"] if K == 3 else [str(g) for g in range(K)])
    ax.set_title(col)
plt.suptitle("Distribuição das principais variáveis em cada grupo do K-Means")
plt.tight_layout()
plt.show()

In [ ]:
for g in range(K):
    paises = df.loc[df["cluster_kmeans"] == g, "country"].sort_values().tolist()
    titulo = NOMES[g] if NOMES else f"Grupo {g}"
    print(f"{titulo} ({len(paises)} países):")
    print(", ".join(paises))
    print()

### 5.3 Aplicação: quais países mais precisam de ajuda?

Dentro do grupo de baixo desenvolvimento, ordenamos os países pela mortalidade infantil e pela renda.
Essa é uma forma simples de transformar o resultado do clustering em uma **recomendação prática**.

In [ ]:
prioridade = (df[df["cluster_kmeans"] == 0]
              .sort_values(["child_mort", "income"], ascending=[False, True])
              [["country", "child_mort", "life_expec", "income", "gdpp"]]
              .head(10)
              .reset_index(drop=True))
prioridade.index += 1
prioridade

## 6. Conclusão

*(Os números exatos estão nas saídas da seção 5.)*

**Análise:**
- O pré-processamento foi decisivo. O K-Means usa **distância euclidiana**, então sem o `log1p` e o `StandardScaler` o PIB per capita
  dominaria o cálculo e os grupos refletiriam apenas a riqueza. Com as variáveis na mesma escala, mortalidade infantil, fertilidade
  e expectativa de vida também passaram a pesar.
- A escolha de **K = 3** veio do método do cotovelo e do silhouette. O silhouette favorece K = 2 (a divisão óbvia entre ricos e pobres),
  mas 3 grupos mantêm uma qualidade próxima e são muito mais informativos para o problema.
- Os grupos encontrados são fáceis de interpretar: países de **baixo desenvolvimento** (alta mortalidade infantil, muitos filhos por mulher,
  baixa renda), **intermediários** e de **alto desenvolvimento** (renda alta, expectativa de vida alta). O PCA mostrou que eles se organizam
  principalmente ao longo de um eixo de "nível de desenvolvimento".
- Com o perfil dos grupos foi possível gerar uma **recomendação prática**: a lista de países prioritários para ajuda humanitária (seção 5.3).
- **Limitações do K-Means:** é preciso **definir K antes** de treinar; o modelo supõe grupos de formato aproximadamente esférico e tamanho parecido;
  e é sensível a valores extremos (por isso o `log1p` ajudou). Os países que ficam na fronteira entre dois perfis podem mudar de grupo com
  pequenas alterações nos dados.
- **Limitação geral:** em aprendizado não supervisionado não existe "resposta certa". As métricas internas ajudam, mas a validação final
  depende de os grupos fazerem **sentido para o problema**, como aconteceu aqui.

## Referências

- Scikit-Learn – Clustering: https://scikit-learn.org/stable/modules/clustering.html
- KMeans: https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html
- Scikit-Learn – Clustering performance evaluation: https://scikit-learn.org/stable/modules/clustering.html#clustering-performance-evaluation
- Kaggle – Unsupervised Learning on Country Data: https://www.kaggle.com/datasets/rohan0301/unsupervised-learning-on-country-data